# From Recorded Session to Training Set: Auditing Robot Logs with FiftyOne

A self-contained, reproducible walkthrough of the full path from raw robot logs
to a training-ready dataset — on **real data** — in three moves:

1. **Ingest** raw MCAP logs into FiftyOne (native, no conversion).
2. **Audit** with embeddings + visualization: see what tasks, conditions, and
   outliers are actually in the logs.
3. **Export** a curated, training-ready dataset with a reproducible recipe.

The data is [`Voxel51/ABC-130k`](https://huggingface.co/datasets/Voxel51/ABC-130k)
— an ungated (~3.5 GB) FiftyOne-native subset of [ABC-130k](https://abc.bot/),
the largest open bimanual robot-teleoperation dataset (Allshire et al., 2026).
The subset is ~40 real teleoperation episodes across ~40 tasks, each an `.mcap`
with a top camera + two wrist cameras and per-arm robot telemetry.

This notebook assumes **no prior FiftyOne install** — the setup cell below
installs everything into your current Python/Jupyter environment. It runs on
macOS, Linux, or Windows. A GPU helps for embeddings but isn't required.

> **One requirement worth knowing up front:** native multimodal MCAP rendering
> needs `fiftyone[multimodal] >= 1.19` and the environment variable
> `VFF_MULTIMODAL=1` set **before** `fiftyone` is imported. Both are handled for
> you below — just run the cells in order.


## 0. Setup — install dependencies

Installs FiftyOne (with the `multimodal` extra) and the packages this notebook
uses, into whatever environment is running this kernel. Safe to re-run; pip
skips anything already present.

If you prefer an isolated environment (recommended but optional), create and
activate a virtualenv **before** launching Jupyter:

```bash
python3 -m venv fiftyone-demo
source fiftyone-demo/bin/activate      # Windows: fiftyone-demo\Scripts\activate
pip install jupyterlab
jupyter lab
```

then run this notebook from that Jupyter. Either way, the install cell below
pulls the rest.

In [ ]:
import sys, subprocess

PACKAGES = [
    "fiftyone[multimodal]>=1.19.0",
    "huggingface_hub",
    "umap-learn",
    "mcap", "mcap-protobuf-support",
    "pillow", "pandas", "matplotlib",
    "torch", "torchvision", "open_clip_torch",  # CLIP embeddings
    "av",                                        # PyAV, for H.264 frame decode
]

# Install into THIS kernel's interpreter.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "pip"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
print("dependencies installed into:", sys.executable)

## 0.1 Set environment variables, then import FiftyOne

These must be set **before** `fiftyone` is imported:

- `VFF_MULTIMODAL=1` — **required** for native multimodal MCAP rendering.
- `HF_XET_HIGH_PERFORMANCE=1` + a download timeout — fast Hugging Face downloads
  with a real progress bar, so the media pull doesn't look hung.
- `OBJC_DISABLE_INITIALIZE_FORK_SAFETY=YES` — silences a harmless macOS warning
  when `opencv` and `av` both bundle FFmpeg (no effect off macOS).

If your kernel has already imported `fiftyone` (e.g. you ran a cell out of
order), restart the kernel and run from here so these take effect.

In [ ]:
import os

os.environ["VFF_MULTIMODAL"] = "1"                        # native multimodal MCAP (required)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"               # fast Xet-backed HF downloads
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"              # fail loudly instead of hanging
os.environ["OBJC_DISABLE_INITIALIZE_FORK_SAFETY"] = "YES" # harmless cv2/av warning (macOS)

# Older huggingface_hub used HF_HUB_ENABLE_HF_TRANSFER; set it only on old
# versions, since newer ones deprecate it in favor of Xet.
try:
    from importlib.metadata import version as _v
    from packaging.version import Version as _V
    if _V(_v("huggingface_hub")) < _V("0.32"):
        os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
except Exception:
    pass

import fiftyone as fo
print("FiftyOne", fo.__version__)

# Heads-up on very new Python versions some deps may not fully support yet.
import sys
if sys.version_info[:2] >= (3, 13):
    print(f"NOTE: Python {sys.version_info.major}.{sys.version_info.minor} detected. "
          "If open_clip/torch import errors appear, use a Python 3.11 environment.")

## 1. Ingest — an MCAP file *is* a sample

In FiftyOne 1.19+, pointing a sample's filepath at a `.mcap` file makes the
dataset's `media_type` automatically `multimodal` — no importer, no conversion,
no frame extraction. The multimodal viewer reads the MCAP directly.

`Voxel51/ABC-130k` is ungated, so this loads with one call — no Hugging Face
token needed. **The first run downloads ~3.5 GB** of episodes and can take a
couple of minutes; it caches afterward, and this cell reuses the dataset on
later runs, so you won't repeat the download.

In [ ]:
import fiftyone.utils.huggingface as fouh

DATASET_NAME = "ABC-130k"
MAX_EPISODES = None   # set e.g. 5 for a quick first pass; None = all ~40

if fo.dataset_exists(DATASET_NAME):
    dataset = fo.load_dataset(DATASET_NAME)
    print(f"reusing cached '{DATASET_NAME}' ({len(dataset)} samples) — no re-download")
else:
    print("first load: downloading ~3.5 GB of episodes (one-time; cached afterward)...")
    dataset = fouh.load_from_hub("Voxel51/ABC-130k", name=DATASET_NAME, persistent=True)
    print(f"loaded '{DATASET_NAME}' ({len(dataset)} samples)")

if MAX_EPISODES:
    dataset = dataset.limit(MAX_EPISODES).clone(name=f"{DATASET_NAME}-subset", persistent=False)
    print("using capped subset:", len(dataset), "samples")

print("media_type:", dataset.media_type)
print("fields:", [f for f in dataset.get_field_schema() if not f.startswith("_")])

## 2. Audit — what is actually in these logs?

Each sample already carries real fields from the source (`task`, `episode_id`,
`station`, `duration_s`, `n_messages`, `camera_topics`, ...), so there's no
parsing or hand-labeling to do. We: (a) summarize the distribution, (b) decode a
representative frame per episode and embed it with CLIP — **attaching the
embedding to the multimodal dataset itself** so the App can do lasso → grid →
video in one session, (c) project with UMAP (the audit act), and (d) read the
distribution.

All heavy steps are cached, so re-running this section is cheap.

### 2a. Distribution summary from the real fields

In [ ]:
import pandas as pd

def gv(s, name):
    try: return s[name]
    except Exception: return None

rows = []
for s in dataset:
    rows.append({
        "episode":    gv(s, "episode_id"),
        "task":       gv(s, "task"),
        "split":      gv(s, "split"),
        "station":    gv(s, "station"),
        "duration_s": gv(s, "duration_s"),
        "n_messages": gv(s, "n_messages"),
        "n_cameras":  len(gv(s, "camera_topics") or []),
    })
meta = pd.DataFrame(rows)
print("episodes:", len(meta), "| distinct tasks:", meta["task"].nunique())
print("\n=== station mix ==="); print(meta["station"].value_counts(dropna=False))
print("\n=== cameras per episode ==="); print(meta["n_cameras"].value_counts())
print("\n=== duration (s) summary ===")
print(meta["duration_s"].describe()[["count","mean","min","max"]].round(2))
meta.head(10)

### 2b. Decode one representative frame per episode

We decode the *middle* frame of each episode's top-camera H.264 stream with
PyAV. Middle (not first): the first frame at t=0 often precedes a keyframe (this
is also why some App grid tiles briefly render black). Frames are cached to disk
with absolute paths and skipped if already decoded, so re-running is instant.

In [ ]:
import numpy as np, io
from pathlib import Path
from PIL import Image
from mcap.reader import make_reader

FRAMES = Path("./_frames").resolve(); FRAMES.mkdir(exist_ok=True)

if "rep_frame" not in dataset.get_field_schema():
    dataset.add_sample_field("rep_frame", fo.StringField)

def top_camera_topic(s):
    cams = None
    try: cams = s["camera_topics"]
    except Exception: cams = None
    if not cams: return None
    for t in cams:
        if "top" in t.lower(): return t
    return cams[0]

def decode_mid_frame(mcap_path, topic):
    try:
        import av
    except ImportError:
        return None
    packets = []
    with open(mcap_path, "rb") as f:
        for schema, channel, message in make_reader(f).iter_messages(topics=[topic]):
            data = message.data
            i = data.find(b"\x00\x00\x00\x01")           # H.264 NAL start code
            if i < 0: i = data.find(b"\x00\x00\x01")
            packets.append(data[i:] if i >= 0 else data)
    if not packets:
        return None
    for codec in ("h264", "hevc"):
        try:
            buf = io.BytesIO(b"".join(packets)); buf.seek(0)
            container = av.open(buf, format=codec)
            frames = [fr for fr in container.decode(video=0)]
            if frames:
                return frames[len(frames)//2].to_ndarray(format="rgb24")
        except Exception:
            continue
    return None

n_ok = n_cached = 0
for s in dataset:
    out = FRAMES / (str(s.id) + ".png")
    if out.exists():
        s["rep_frame"] = str(out); s.save(); n_ok += 1; n_cached += 1
        continue
    topic = top_camera_topic(s)
    arr = decode_mid_frame(s.filepath, topic) if topic else None
    if arr is not None:
        Image.fromarray(arr).save(out)
        s["rep_frame"] = str(out); n_ok += 1
    else:
        s["rep_frame"] = None
    s.save()

print(f"representative frames: {n_ok}/{len(dataset)} ({n_cached} from cache)")
if n_ok == 0:
    print("If 0: ensure PyAV (`av`) is installed; embeddings fall back to histograms.")

### 2c. Embed the frames and attach to the multimodal dataset

CLIP needs an image to embed, so we embed the decoded frames, then attach each
embedding back onto the multimodal dataset (keyed by episode). Keeping the
embedding *on the multimodal dataset* is what lets the App do lasso → grid →
video in one session. Falls back to a color histogram if CLIP isn't available.

In [ ]:
import fiftyone.brain as fob

FRAMES_DS = "abc130k_frames"   # temporary image dataset, just for embedding
if fo.dataset_exists(FRAMES_DS):
    fo.delete_dataset(FRAMES_DS)
frames = fo.Dataset(FRAMES_DS)

ep_by_framepath = {}
frame_samples = []
for s in dataset:
    fp = s.get_field("rep_frame")
    if fp and Path(fp).exists():
        frame_samples.append(fo.Sample(filepath=fp))
        ep_by_framepath[fp] = str(s.get_field("episode_id"))
assert frame_samples, "No decoded frames to embed (see previous cell)."
frames.add_samples(frame_samples)

def compute_embeddings(ds):
    try:
        import fiftyone.zoo as foz
        model = foz.load_zoo_model("open-clip-torch")   # CLIP ViT-B/32
        ds.compute_embeddings(model, embeddings_field="embedding")
        return "clip"
    except Exception as e:
        print("CLIP unavailable, color-histogram fallback:", e)
        for s in ds:
            im = np.asarray(Image.open(s.filepath).convert("RGB").resize((64,64)))
            h = np.concatenate([np.histogram(im[...,c],bins=16,range=(0,255))[0]
                                for c in range(3)]).astype(float)
            h /= h.sum()+1e-9; s["embedding"]=h.tolist(); s.save()
        return "hist"

emb_kind = compute_embeddings(frames)

emb_by_ep = {}
for s in frames:
    ep = ep_by_framepath.get(s.filepath)
    if ep is not None:
        emb_by_ep[ep] = s["embedding"]

if "embedding" not in dataset.get_field_schema():
    dataset.add_sample_field("embedding", fo.VectorField)

n_attached = 0
for s in dataset:
    e = emb_by_ep.get(str(s.get_field("episode_id")))
    if e is not None:
        s["embedding"] = e; s.save(); n_attached += 1

print("embeddings:", emb_kind, "| attached to", n_attached, "multimodal samples")

### 2d. The audit act — project the embedding space (UMAP)

UMAP's defaults can hang on tiny datasets, so we set `num_neighbors` below the
sample count and a fixed `seed` for reproducibility. The run is cached under a
brain key and reused on re-run.

In [ ]:
BRAIN_KEY = "abc130k_viz"
n = len(dataset)

if BRAIN_KEY in dataset.list_brain_runs():
    res = dataset.load_brain_results(BRAIN_KEY)
    print("reusing cached UMAP run")
else:
    res = fob.compute_visualization(
        dataset, embeddings="embedding", method="umap",
        brain_key=BRAIN_KEY,
        num_neighbors=min(10, max(2, n - 1)),   # must be < n_samples
        seed=51, verbose=True,
    )
print("projection points:", len(res.current_points))

### 2e. Static plot of the projection

A quick two-panel scatter (by task, by station) saved to `audit_embeddings.png`.
The interactive version — where you lasso to select — is in the App (section 4).

In [ ]:
import matplotlib.pyplot as plt
pts = np.array(res.current_points)

tasks    = dataset.values("task")
stations = dataset.values("station")

def scatter(labels, ax, title, legend=True):
    labels = [l if l is not None else "unknown" for l in labels]
    uniq = sorted(set(labels))
    for lab in uniq:
        m = np.array([x == lab for x in labels])
        ax.scatter(pts[m,0], pts[m,1], s=55, alpha=.85, label=lab)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    if legend and len(uniq) <= 12:
        ax.legend(fontsize=6, loc="best")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
scatter(tasks,    axes[0], f"by TASK ({len(set(tasks))} tasks)", legend=False)
scatter(stations, axes[1], "by STATION (mono=H.264 / stereo=H.265)")
fig.suptitle("Audit: what is actually in the ABC-130k subset — one point per real episode",
             fontsize=12)
plt.tight_layout()
plt.savefig("audit_embeddings.png", dpi=120, bbox_inches="tight")
plt.show()
print("saved audit_embeddings.png")
print("Note: ~1 episode per task here, so the TASK panel is a COVERAGE map, not "
      "tight clusters. For the cluster picture, load several episodes per task.")

## 3. Export — a curated set with a recipe, not a vibe

Curation is a filter over the dataset, so the filter *is* the specification.
This subset is all one station type, so we split on real signal — **duration** —
keeping episodes at or above the median for training and holding shorter ones
out for review (short episodes are likelier to be truncated/degenerate). Edit
the rule to fit your task.

Exports three artifacts: a lossless FiftyOne dataset (round-trips every field +
the MCAP paths), a portable `manifest.csv`, and a `recipe.json` capturing the
exact filter so the split is reproducible.

In [ ]:
from fiftyone import ViewField as F
import json

durations = [d for d in dataset.values("duration_s") if d is not None]
median_duration = float(np.median(durations))
print(f"median duration = {median_duration:.2f}s  (train >= median, review < median)")

train_view = (
    dataset
    .match(F("rep_frame") != None)
    .match(F("duration_s") != None)
    .match(F("duration_s") >= median_duration)
)
review_view = dataset.match(
    (F("duration_s") == None) | (F("duration_s") < median_duration) | (F("rep_frame") == None)
)

print(f"training episodes : {len(train_view)}")
print(f"review holdout    : {len(review_view)}")

dataset.untag_samples(["train", "review_holdout"])
train_view.tag_samples("train"); review_view.tag_samples("review_holdout")

EXPORT = Path("./training_ready_export").resolve(); EXPORT.mkdir(exist_ok=True)
train_view.export(export_dir=str(EXPORT/"fiftyone_dataset"),
                  dataset_type=fo.types.FiftyOneDataset, export_media=False)

manifest = []
for split, view in [("train", train_view), ("review_holdout", review_view)]:
    for s in view:
        manifest.append({
            "episode_mcap": s.filepath,
            "split": split,
            "task": s.get_field("task"),
            "episode_id": s.get_field("episode_id"),
            "station": s.get_field("station"),
            "duration_s": s.get_field("duration_s"),
            "n_cameras": len(s.get_field("camera_topics") or []),
        })
pd.DataFrame(manifest).to_csv(EXPORT/"manifest.csv", index=False)
(EXPORT/"recipe.json").write_text(json.dumps({
    "source": "Voxel51/ABC-130k",
    "train_filter": ["rep_frame present", f"duration_s >= median({median_duration:.2f})"],
    "review_holdout_filter": ["duration_s < median OR missing frame/duration"],
    "median_duration_s": round(median_duration, 2),
    "embedding_method": emb_kind,
    "n_train": len(train_view), "n_review_holdout": len(review_view),
}, indent=2))

print("\nexported to:", EXPORT)
for p in sorted(EXPORT.rglob("*")):
    if p.is_file(): print("  ", p.relative_to(EXPORT))

## 4. Explore in the App — and demo the audit

Launch the multimodal viewer. Because the embeddings are on this dataset and
UMAP ran on it, the **Embeddings panel works in the same App** as the multimodal
video — so the whole audit demo happens in one place.

**The audit, three ways to see it in the App:**

1. **Embeddings panel (the map).** Click the **`+`** next to the *Samples* tab →
   **Embeddings** → pick the `abc130k_viz` run. Every episode is one point. Use
   *color by* → `task` (coverage) or `station`. **Lasso** a region and the grid
   filters to those episodes — the lasso is the curation gesture. Click a point
   to open the tiled camera + telemetry viewer.
2. **Distribution (the table/chart).** Expand the chevron next to any sidebar
   field (`task`, `station`, `duration_s`) for live per-value counts, or add a
   **Histograms** panel via the **`+`** for a chart of any field.
3. **The split (first-class metadata).** Scroll the sidebar to the **TAGS**
   section at the top. Check **`train`** — the grid (and any Histograms panel)
   filters to the 20 curated episodes; check **`review_holdout`** to see what
   you set aside. These are the same splits written to `manifest.csv`.

> **Want tight clusters instead of a coverage map?** This subset is ~1 episode
> per task. Load several episodes across a few tasks (see the loader repo linked
> in the recap), re-run section 2, and you'll get dense, lasso-able clusters.

**Grid rendering note:** keep the grid on `filepath` (default). The App decodes
each episode's H.264 in the browser for grid tiles; a tile can briefly show
black when its first frame precedes a keyframe, but it resolves as byte-range
reads cache — no fix needed. If you want static thumbnails instead, the
commented block uses the absolute-path frames from 2b.

In [ ]:
# Keep the grid on the video (default, renders correctly):
dataset.app_config.media_fields = ["filepath"]
dataset.app_config.grid_media_field = "filepath"
dataset.app_config.media_fallback = False
dataset.save()

# --- OPTIONAL: static image thumbnails in the grid instead of live video ---
# rep_frame holds ABSOLUTE paths (set in 2b), so this is safe for the App server:
# dataset.app_config.media_fields = ["filepath", "rep_frame"]
# dataset.app_config.grid_media_field = "rep_frame"
# dataset.app_config.media_fallback = True
# dataset.save()

session = fo.launch_app(dataset)

# If you apply tags or config AFTER launching, call session.refresh() (or reload
# the browser tab) so the App picks them up — e.g. the train/review_holdout tags
# won't appear under the sidebar TAGS section until a refresh.
session.refresh()
session

## Recap: the defined path

1. **Ingest** — pointed samples at `.mcap` files; `media_type` inferred as
   `multimodal`; no conversion. One sample per recorded episode.
2. **Audit** — decoded a representative frame per episode, embedded with CLIP,
   attached the embeddings to the multimodal dataset, and projected with UMAP.
   The interactive Embeddings panel + sidebar distributions answer "what's
   actually in these logs," and the lasso selects candidate subsets.
3. **Export** — expressed a curation decision as a filter, tagged the split, and
   exported a lossless dataset + `manifest.csv` + `recipe.json`. Reproducible
   because the decision is written down as the thing that produced the data.

**Scaling up:** to pull more of ABC-130k (specific tasks, several episodes per
task for real clusters, the full val split, or the gated `XDOF/ABC-130k`
source), use the loader repo
[`github.com/Burhan-Q/fiftyone-abc130k`](https://github.com/Burhan-Q/fiftyone-abc130k)
(`abc130k.py`), which supports `--max-tasks`, `--episodes-per-task`,
`--budget-gb`, and `--dry-run`.

*Dataset: ABC-130k (Allshire et al., 2026), Apache-2.0. `Voxel51/ABC-130k` is an
unofficial FiftyOne-repackaged subset of the validation split; cite the original
paper.*